# Week04 Capstone project - svm_boundary_sampling

## Strategy
For Week 4, the query policy treats each function as a small **good vs bad region classification** problem.

The idea is:
1. combine the starter `.npy` data with Weeks 1–3 from `Week04.csv`
2. convert outputs into binary labels using a per-function quantile threshold
3. train an **RBF SVM** to learn the rough boundary between promising and weak regions
4. sample a large candidate pool on a **0.01 grid** for human-clean points
5. prefer points that are:
   - near the SVM decision boundary (informative)
   - on the predicted good side of the boundary
   - supported by a GP tie-breaker using predicted mean and uncertainty
   - not too close to previously evaluated points

This gives a  **boundary-sampling** strategy: it is still exploratory, but not random wandering.


In [1]:
import numpy as np
import pandas as pd
import warnings
from pathlib import Path
from sklearn.svm import SVC
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import ConstantKernel, RBF, WhiteKernel
from sklearn.exceptions import ConvergenceWarning


/Users/sundarid/opt/anaconda3/lib/python3.9/site-packages/scipy/__init__.py:146: UserWarning: A NumPy version >=1.16.5 and <1.23.0 is required for this version of SciPy (detected version 1.26.4
  warnings.warn(f"A NumPy version >={np_minversion} and <{np_maxversion}"


In [3]:
data_dir = Path('data')
history = pd.read_csv(data_dir / 'weekly_results/Week04.csv')
history.head(12)


,week,function,d,x1,x2,x3,x4,x5,x6,x7,x8,y
0,1,1,2,0.759779,0.804108,NaN,NaN,NaN,NaN,NaN,NaN,3.100722e-34
1,1,2,2,0.717915,0.002064,NaN,NaN,NaN,NaN,NaN,NaN,5.697925e-01
2,1,3,3,0.994942,0.051967,0.792526,NaN,NaN,NaN,NaN,NaN,-1.518442e-01
3,1,4,4,0.404477,0.413254,0.303108,0.434359,NaN,NaN,NaN,NaN,-7.158151e-01
4,1,5,4,0.940282,0.064909,0.998637,0.996528,NaN,NaN,NaN,NaN,3.653951e+03
5,1,6,5,0.379776,0.684419,0.068171,0.984146,0.290503,NaN,NaN,NaN,-1.362062e+00
6,1,7,6,0.015298,0.590564,0.658382,0.751493,0.425786,0.776978,NaN,NaN,2.395347e-01
7,1,8,8,0.034492,0.437681,0.011459,0.323393,0.989664,0.045780,0.097175,0.702465,9.589938e+00
8,2,1,2,0.803235,0.719438,NaN,NaN,NaN,NaN,NaN,NaN,-1.512885e-27
9,2,2,2,0.957387,0.923033,NaN,NaN,NaN,NaN,NaN,NaN,-7.246455e-02


In [4]:
def load_initial(function_id):
    X0 = np.load(data_dir / f'initial_data/F{function_id}_initial_inputs.npy')
    y0 = np.load(data_dir / f'initial_data/F{function_id}_initial_outputs.npy')
    return X0, y0

def load_combined(function_id, history_df):
    X0, y0 = load_initial(function_id)
    rows = history_df[history_df['function'] == function_id].sort_values('week')
    d = int(rows['d'].iloc[0])
    Xw = rows[[f'x{i}' for i in range(1, d + 1)]].to_numpy(dtype=float)
    yw = rows['y'].to_numpy(dtype=float)
    X = np.vstack([X0, Xw])
    y = np.concatenate([y0, yw])
    return X, y, d

def format_point(x):
    return '-'.join(f'{float(v):.6f}' for v in x)


In [14]:
def suggest_week4_point(function_id, history_df, *, seed=123, n_candidates=50000, quantile=0.80, grid_step=0.01):
    X, y, d = load_combined(function_id, history_df)

    # Convert regression targets into a simple good/bad label.
    threshold = np.quantile(y, quantile)
    labels = (y >= threshold).astype(int)
    if labels.min() == labels.max():
        threshold = np.median(y)
        labels = (y >= threshold).astype(int)

    rng = np.random.RandomState(seed + function_id)
    candidates = rng.uniform(0.0, 1.0, size=(n_candidates, d))
    candidates = np.round(candidates / grid_step) * grid_step
    candidates = np.clip(candidates, 0.0, 1.0)
    candidates = np.unique(candidates, axis=0)

    # Normalised distance to previously evaluated points.
    distances = np.sqrt(((candidates[:, None, :] - X[None, :, :]) ** 2).sum(axis=2)) / np.sqrt(d)
    min_dist = distances.min(axis=1)

    # SVM boundary model.
    svm = make_pipeline(
        StandardScaler(),
        SVC(kernel='rbf', C=2.0, gamma='scale')
    )
    svm.fit(X, labels)
    decision = svm.decision_function(candidates)
    pred_good = (decision > 0).astype(float)

    # GP tie-breaker: predicted mean + uncertainty.
    kernel = (
        ConstantKernel(1.0, (0.1, 10.0))
        * RBF(length_scale=np.ones(d), length_scale_bounds=(0.05, 2.0))
        + WhiteKernel(noise_level=1e-5, noise_level_bounds=(1e-8, 1e-2))
    )
    gp = GaussianProcessRegressor(
        kernel=kernel,
        normalize_y=True,
        random_state=seed + function_id,
        n_restarts_optimizer=1,
    )
    with warnings.catch_warnings():
        warnings.filterwarnings('ignore', category=ConvergenceWarning)
        gp.fit(X, y)

    mean, std = gp.predict(candidates, return_std=True)
    mean_z = (mean - np.mean(y)) / (np.std(y) + 1e-9)
    std_z = std / (np.max(std) + 1e-9)

    # Boundary-sampling score
    score = (
        0.60 * pred_good
        - 0.80 * np.abs(decision)
        + 0.15 * mean_z
        + 0.15 * std_z
        + 0.25 * min_dist
    )

    best_idx = np.argmax(score)
    x_best = candidates[best_idx]

    return {
        'function': function_id,
        'd': d,
        'x': x_best,
        'formatted': format_point(x_best),
        'threshold': float(threshold),
        'mean_pred': float(mean[best_idx]),
        'std_pred': float(std[best_idx]),
        'decision_value': float(decision[best_idx]),
        'min_dist': float(min_dist[best_idx]),
        'kernel': str(gp.kernel_),
    }
